# Qwen2.5-0.5B Fine-Tuning — Section 8.1 Tax Law
**Before running:** Kaggle > Settings > Accelerator = `GPU T4 x2` | Internet = `ON`

---
## STEP 1 — Install (run this first, wait for kernel restart)

In [ ]:
import subprocess

def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print('FAIL:', r.stderr[-200:])
    else:
        print('OK:', cmd[:60])

# 1. Remove all conflicting packages
run('pip uninstall -y unsloth unsloth-zoo transformers trl bitsandbytes triton peft accelerate')

# 2. Install CUDA-compatible bitsandbytes
run('pip install bitsandbytes==0.43.3 --prefer-binary')

# 3. Install triton (version with triton.ops)
run('pip install triton==2.1.0')

# 4. Install transformers (compatible version)
run('pip install transformers==4.45.2 tokenizers==0.20.3 --no-deps')

# 5. Install training libraries
run('pip install peft==0.13.2')
run('pip install accelerate==1.0.1')
run('pip install trl==0.11.4')
run('pip install datasets==3.0.2')

# 6. Install Unsloth
run('pip install unsloth-zoo==2024.11.9')
run('pip install unsloth==2024.11.9 --no-deps')

print('\n✅ DONE! Restarting kernel...')
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

---
## STEP 2 — Check imports (run after kernel restarts)

In [ ]:
import transformers, trl, unsloth
print('transformers :', transformers.__version__)
print('trl          :', trl.__version__)
print('unsloth      :', unsloth.__version__)
print('\n✅ All imports OK!')

---
## STEP 3 — Load Model

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = 'unsloth/Qwen2.5-0.5B-Instruct',
    max_seq_length = 512,
    load_in_4bit   = True,
    dtype          = None,
)
print('✅ Model loaded!')

---
## STEP 4 — Apply LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    target_modules = ['q_proj','k_proj','v_proj','o_proj',
                      'gate_proj','up_proj','down_proj'],
    lora_alpha     = 16,
    lora_dropout   = 0,
    bias           = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state   = 42,
)
print('✅ LoRA applied!')

---
## STEP 5 — Prepare Dataset

In [ ]:
from datasets import Dataset

SYS = 'Answer ONLY using Section 8.1. Do not go outside the section.'

raw_data = [
    {'user': 'What is Section 8.1 of the Income Tax Assessment Act 1997 about?',
     'assistant': 'Section 8.1 is about general deductions. It explains when you can and cannot deduct a loss or outgoing from your assessable income.'},

    {'user': 'Under Section 8.1, when can you deduct a loss or outgoing from your assessable income?',
     'assistant': 'You can deduct a loss or outgoing from your assessable income if it is incurred in gaining or producing your assessable income, or if it is necessarily incurred in carrying on a business for the purpose of gaining or producing your assessable income.'},

    {'user': 'What are the two positive conditions under Section 8.1(1) that allow a deduction?',
     'assistant': 'The two positive conditions are: (a) the loss or outgoing is incurred in gaining or producing your assessable income, or (b) it is necessarily incurred in carrying on a business for the purpose of gaining or producing your assessable income.'},

    {'user': 'Under Section 8.1(2), what types of losses or outgoings CANNOT be deducted?',
     'assistant': 'Under Section 8.1(2), you cannot deduct a loss or outgoing if: (a) it is of a capital or capital nature, (b) it is of a private or domestic nature, (c) it is incurred in relation to gaining or producing exempt income or non-assessable non-exempt income, or (d) a provision of the Act prevents you from deducting it.'},

    {'user': 'Can you deduct a loss of a capital nature under Section 8.1?',
     'assistant': 'No. Under Section 8.1(2)(a), you cannot deduct a loss or outgoing to the extent that it is a loss or outgoing of capital, or of a capital nature.'},

    {'user': 'Can you deduct a private or domestic expense under Section 8.1?',
     'assistant': 'No. Under Section 8.1(2)(b), you cannot deduct a loss or outgoing to the extent that it is a loss or outgoing of a private or domestic nature.'},

    {'user': "What is a 'general deduction' according to Section 8.1(3)?",
     'assistant': 'According to Section 8.1(3), a loss or outgoing that you can deduct under this section is called a general deduction.'},

    {'user': 'What happens if you receive insurance or indemnity for a loss that you deducted under Section 8.1?',
     'assistant': 'If you receive an amount as insurance, indemnity or other recoupment of a loss or outgoing that you can deduct under Section 8.1, the amount may be included in your assessable income under Subdivision 20-A.'},

    {'user': 'What does Division 35 do in relation to Section 8.1?',
     'assistant': 'Division 35 prevents losses from non-commercial business activities that may contribute to a tax loss from being offset against other assessable income.'},

    {'user': 'Can you deduct an outgoing incurred in relation to gaining exempt income under Section 8.1?',
     'assistant': 'No. Under Section 8.1(2)(c), you cannot deduct a loss or outgoing to the extent that it is incurred in relation to gaining or producing your exempt income or your non-assessable non-exempt income.'},
]

def format_chat(ex):
    msgs = [
        {'role': 'system',    'content': SYS},
        {'role': 'user',      'content': ex['user']},
        {'role': 'assistant', 'content': ex['assistant']},
    ]
    return {'text': tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)}

dataset = Dataset.from_list(raw_data).map(format_chat)
print(f'✅ Dataset ready — {len(dataset)} examples')

---
## STEP 6 — Train

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = dataset,
    dataset_text_field = 'text',
    max_seq_length     = 512,
    dataset_num_proc   = 2,
    packing            = False,
    args = TrainingArguments(
        num_train_epochs            = 10,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps                = 5,
        learning_rate               = 2e-4,
        weight_decay                = 0.01,
        fp16       = not is_bfloat16_supported(),
        bf16       = is_bfloat16_supported(),
        logging_steps  = 5,
        save_strategy  = 'no',
        output_dir     = '/kaggle/working/output',
        optim          = 'adamw_8bit',
        seed           = 42,
        report_to      = 'none',
    ),
)

print('🚀 Training started... (~10 mins)')
stats = trainer.train()
print(f'\n✅ Done! Loss: {stats.metrics["train_loss"]:.4f}')
print('💡 Loss < 0.1 = model learned dataset perfectly!')

---
## STEP 7 — Test Model

In [ ]:
FastLanguageModel.for_inference(model)

def ask(q):
    msgs = [
        {'role': 'system', 'content': SYS},
        {'role': 'user',   'content': q},
    ]
    inp = tokenizer.apply_chat_template(
        msgs, tokenize=True,
        add_generation_prompt=True,
        return_tensors='pt'
    ).to('cuda')
    out = model.generate(
        inp, max_new_tokens=150,
        temperature=0.1, do_sample=True,
        repetition_penalty=1.1
    )
    return tokenizer.decode(out[0][inp.shape[1]:], skip_special_tokens=True).strip()

questions = [
    'What is Section 8.1 about?',
    'Can you deduct a private expense?',
    'What is a general deduction?',
    'What does Division 35 do?',
]

for q in questions:
    print(f'Q: {q}')
    print(f'A: {ask(q)}')
    print('-' * 50)

---
## STEP 8 — Save Model

In [ ]:
PATH = '/kaggle/working/qwen_section81'
model.save_pretrained(PATH)
tokenizer.save_pretrained(PATH)
print(f'✅ Model saved → {PATH}')
print('📥 Download from Kaggle Output tab on the right')